# TMFT End-to-End Experiment

Run this notebook from the uploaded `tmft_project/` directory. It prepares real PII-containing Enron splits, trains all five methods, and computes TER, SER, perplexity, MDP, and two MIA AUC metrics.

## 0. Verify Project Structure

In [ ]:
from pathlib import Path
import os, sys, platform

PROJECT_ROOT = Path.cwd()
required = [
    'configs/config.yaml', 'src/data_prep.py', 'src/train.py',
    'src/masking.py', 'src/evaluate_pii.py', 'src/evaluate_ppl.py',
    'src/evaluate_mia.py', 'src/plot_results.py', 'main.py', 'requirements.txt'
]
missing = [path for path in required if not (PROJECT_ROOT / path).exists()]
if missing:
    raise FileNotFoundError('Open the notebook inside the full tmft_project directory. Missing: ' + ', '.join(missing))
print('Project:', PROJECT_ROOT)
print('Python:', sys.version)
print('Platform:', platform.platform())

## 1. Install Dependencies
Run once in a fresh Vessel workspace, then restart the kernel.

In [ ]:
%pip install -U pip
%pip uninstall -y transformers peft accelerate tokenizers huggingface_hub datasets
%pip install -r requirements.txt
!python -m spacy download en_core_web_sm
print('Restart the kernel now, then continue from the next cell.')

## 2. Imports and GPU Check

In [ ]:
import os
os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')

import pandas as pd
import torch
from src.data_prep import prepare_experiment_data
from src.plot_results import plot_results
from src.train import METHODS, load_config, train_model, upload_to_huggingface
from main import run_eval

print('Methods:', METHODS)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Smoke-Test Data Preparation

In [ ]:
smoke_config = load_config('configs/config.yaml')
smoke_config.update({
    'max_train_samples': 100,
    'max_prepared_samples': 80,
    'max_eval_samples': 20,
    'prepared_data_dir': './data/smoke_processed',
    'pii_eval_path': './data/smoke_pii_eval.json',
    'num_epochs': 0.02,
    'batch_size': 1,
    'gradient_accumulation_steps': 1,
    'fp16': False,
    'output_dir': './results/smoke',
})
smoke_splits, smoke_eval_path = prepare_experiment_data(smoke_config, force=True)
print(smoke_splits)
print('Real evaluation records:', smoke_eval_path)

## 4. Smoke-Test Training

In [ ]:
trainer, tokenizer, smoke_output = train_model(
    smoke_config,
    method='tmft_ner',
    train_dataset=smoke_splits['train'],
    eval_dataset=smoke_splits['validation'],
)
print('Saved:', smoke_output)

## 5. Final Data Preparation
This rebuilds real PII-containing train/validation/test splits and automatically creates `data/pii_eval.json`.

In [ ]:
final_config = load_config('configs/config.yaml')
final_config['fp16'] = False
final_splits, eval_path = prepare_experiment_data(final_config, force=True)
final_config['text_column'] = 'text'
print(final_splits)
print('PII eval file:', eval_path)
print('Train sample:', final_splits['train'][0]['text'][:500])

## 6. Train Five Conditions
For a time-limited pilot, use the first three methods. The final report should use all five.

In [ ]:
methods_to_run = ['baseline', 'rmft', 'tmft_ner', 'tmft_mia', 'tmft_combined']
train_outputs = {}
for method in methods_to_run:
    print(f'\n===== TRAIN: {method} =====')
    trainer, tokenizer, output_dir = train_model(
        final_config,
        method=method,
        train_dataset=final_splits['train'],
        eval_dataset=final_splits['validation'],
    )
    train_outputs[method] = str(output_dir)
train_outputs

## 7. Evaluate Privacy and Utility
Computes TER, SER, held-out PPL/MDP, Loss-MIA AUC, and Min-K MIA AUC.

In [ ]:
results_df = run_eval(final_config, 'all', final_splits, eval_path)
results_df

## 8. Generate Submission Figures

In [ ]:
csv_path = Path(final_config['results_table_dir']) / 'main_results.csv'
figure_paths = plot_results(csv_path, 'results/figures')
print('Table:', csv_path)
print('Figures:', figure_paths)

## 9. Hugging Face Upload
Log in with `huggingface-cli login` first. Upload one validated checkpoint at a time.

In [ ]:
HF_REPO_ID = 'YOUR_USERNAME/tmft-pythia-160m-tmft-combined'
model_dir = train_outputs.get('tmft_combined', 'results/tmft_combined')
print('Ready:', model_dir, '->', HF_REPO_ID)
# upload_to_huggingface(model_dir, repo_id=HF_REPO_ID, private=True)